In [ ]:
# Install latest version from local files (only necessary after pulling updated version from Git).
# Restart Jupyter kernel when install has finished.
%pip install -e ..

In [ ]:
# Some Windows installations require explicitly installing mne (required only once; restart Jupyter kernel after installing).
%pip install mne mne_bids

In [ ]:
from biopap.pipeline import PipelineStep, Pipeline

from biopap.modules.load_data import LoadData
from biopap.modules.anonymize import Anonymize
from biopap.modules.rereference import Rereference
from biopap.modules.eog_channels import EOGChannels
from biopap.modules.iir_filter import IIRFilter
from biopap.modules.downsample import Downsample
from biopap.modules.segmentation import Segmentation
from biopap.modules.plot import Plot

In [ ]:
from pathlib import Path
from pprint import pprint

In [ ]:
# To avoid issues with local settings and GitHub updates, create a small notebook
# called 'my_config.ipynb' to hold variables that are likely to change locally. 
# In the same folder as this file, there is an example notebook called
# 'example.my_config.ipynb' that can serve as a template.

config_notebook = './my_config.ipynb'

if Path(config_notebook).exists():
    %run {config_notebook}
else:
    data_file = '/biopap/data/example.bdf'
    config_file = './data/pipeline.yaml'

if not Path(config_file).exists:
    print(f"Config file {config_file} does not exist.")

if not Path(data_file).exists:
    print(f"Data file {data_file} does not exist.")

### About modules

When instantiated, all processing modules receive the data to be processed as RawEDF-type object, which is assigned to an attribute labeled `raw`. Once done processing, the processed data is available as an attribute labeled `processed` of the module. The processed data has the same format as the raw input, so the output of one module can become the input of the next, thus allowing you to easily chain modules together.

The only exception is the Load module, which reads the initial data from disk without any further processing, and therefore only has raw data.

### Loading data

In [ ]:
# Read the data file and load it into memory.
load = LoadData(file_path=data_file)

In [ ]:
# use the Plot-module to inspect the data
plt = Plot(raw=load.raw, title="Initial state")

### Anonymization

In [ ]:
# defaults: Participant='0'; Experimenter='mne_anonymize'
ano = Anonymize(raw=load.raw, daysback=100, experimenter='J. Jansen', participant_id='AB00123')
ano.processed.info

### Rereferencing

In [ ]:
ref = Rereference(raw=ano.processed, ref_channels=['EXG1', 'EXG2'])
ref.processed.info

### hEOG & vEOG

In [ ]:
eog = EOGChannels(
    raw=ref.processed,
    ref_channels=['hEOG', 'vEOG'],
    anode_channels=['EXG3','EXG5'],
    cathode_channels=['EXG4','EXG6'])

In [ ]:
print(eog.processed.info)

### Filtering

In [ ]:
flt = IIRFilter(
    raw=eog.processed,
    l_freq=0.1,
    h_freq=35)

In [ ]:
plt = Plot(raw=flt.processed)

In [ ]:
# Alternatively, save a plot of the current state by specifying save folder. The screenshot will take the basename of the original
# source file, optionally taking a postfix to distinguish between multiple screenshots on separate points in the pipeline.
# This step is not interactive, so can also be used in a pipeline intended to run unattended.
# N.B., showing the interactive plot and saving cannot be done at the same time, but you can add multiple subsequent Plot-steps.
_ = Plot(raw=flt.processed, save_folder='./', save_postfix='IRR')

### Downsampling

In [ ]:
dwn = Downsample(
    raw=flt.processed,
    sfreq=500)

In [ ]:
print(dwn.processed)
print(dwn.events)